# # AdaBoost
Ensemble methods are a popular approach to construct strong estimators from learnes barely different that random estimators, in previous guides we have covered some bagging algorithms like random forests, extremly randomized trees and gradient boosted trees (GBT) where we defined the concept of boosting. But GBT weren't the first approach in applying boosting strategy, there is and older but efficient algorithm, nowadays isn't widely used as end of 90's and begins of 2000's however is a powerful approach for training robust estimators from weak learners

## Boosting
I don't want repeat my words so I let you definition given at [Gradient boosting trees guide](https://www.kaggle.com/code/criser2013/gradient-boosting-trees) about boosting approach.

>Boosting is an ensemble technique that builds a strong model by combining multiple weak learners sequentially.
>This is different from bagging, where individual learners are trained independently—often on different samples of the training data—and their predictions are then aggregated.
>In boosting, models are trained sequentially. Each new learner is added to improve the predictions made by the ensemble built so far. In other words, instead of training every tree independently, boosting uses the information from previous iterations to determine what the next tree should learn.
>The final model can be viewed as a weighted sum of the predictions produced by the individual learners.

## Adaboost
Adaboost consists on taking a base estimator, a model that is barely better than a random estimator then training other estimators that focuses about learning mispredicted instances and the process continues until reaching the desired amount of estimators. Each estimator has assigned a "weight" value that indicates how good were its results at training stage. Classical implementation uses decision trees, but Scikit library allows to use almost **any** estimator for regression or classification tasks, the estimator choosen must support sample weighting. The learning algorithm is described as follows.

### Training algorithm
#### Classification tasks
Let's define:
- $D = \lbrace (x_i, y_i) \ i \in {1, 2, ..., n} \rbrace$ where $n$ is the number of samples the training dataset.
- $T^{(0)}$ the base estimator, a weak learner that supports sample weighting.
- $\hat{y}_{i}^{j}$ the output of estimator $j$ for instance $i$.
- $K$ as the number of classes.

the Stagewise Additive Modeling using Multi-class Exponential loss function (SAMME) proceeds as follows:

1. Set initial weights $w_i$ for each sample in the dataset as follows:
$$
    w_i = \frac{1}{n}
$$
2. Fit $T^{(0)}$ using the training dataset and the computed weights.
3. Compute the error of the estimator for each instance.
$$
    \mathbf{I}(y_i = \hat{y}_{i}^{(k)}) = \begin{cases}
        1 & \text{if } y_i = \hat{y}_{i}^{(k)} \\
        0 & \text{otherwise}
    \end{cases}
$$
<br/>
$$
    err^{(j)} = \frac{\sum_{i=1}^{n} w_i \cdot \mathbf{I}(y_i \not = \hat{y}_{i}^{(k)})}{\sum_{i=1}^{n} w_i}
$$
3. Compute the weight $\alpha^{(0)}$ for the estimator as follows:
$$
    \alpha^{(j)} = log(\frac{1-err^{(j)}}{err^{(j)}}) + log(K - 1)
$$
4. Compute and set the value of the weights.
$$
    w_i = w_i \cdot exp(\alpha^{(m)} \cdot \mathbf{I}(y_i \not = \hat{y}_{i}^{(k)}))
$$
5. Repeat steps 2 to 4 until reaching the number of desired estimators.


## Implementation
### Classification
Works like random forests models that uses a majority voting system but it weights the learners outcome using $\alpha$ coefficients of each estimator. Mathematically the formulation is:

$$
        \hat{y}(x) = \operatorname*{arg\,max}_{k \in \{1,\ldots, K \}} \sum_{j=1}^{M} \alpha^{(m)} \cdot \mathbf{I}(\hat{y}^{(j)}(x) = k)
$$

SAMME works for binary and multiclass tasks.

In [2]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.tree import ExtraTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = AdaBoostClassifier(estimator=ExtraTreeClassifier(max_depth=7, splitter="best", criterion="gini"),
                                          n_estimators=100, learning_rate=0.5, random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

Accuracy: 90.000000 %
Weighted F1-Score: 89.950459 %


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.tree import ExtraTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = AdaBoostRegressor(estimator=ExtraTreeRegressor(max_depth=7, splitter="best", criterion="gini"),
                                    n_estimators=100, learning_rate=0.5, random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")


## Hyperparameters
- **`estimator`**: The base estimator from which the boosted ensemble is built. Must support sample weighting
- **`n_estimators`**: Number of estimators to build.
- **`learning_rate`**: Weight applied to each classifier at each boosting iteration.
- **`loss`**: The loss function to use when updating the weights after each boosting iteration.

## Advantages
- Faster than GBT because calculate sample and tree weights is lifter than calculate the gradient for each tree.
- The adaptive learning pattern dismiss overfitting because the model focus learning about its mistakes.
- Has a mechanism to find "most important" features in our dataset.

## Disadavantages
- Focus on noisy samples as they're harder to predict.
- Only works for sequentially trained estimators because it needs the errors of the previous one to build the next one.
- Requires a deep testing on learning rate because high values can overfit the model but low ones can require more estimators to achieve a good performance.